In [1]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = SparkSession.builder \
    .master("spark://spark-master:7077") \
    .appName("Partition") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4") \
    .config("spark.sql.extensions","io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog","org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint","http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin" ) \
    .config("spark.hadoop.fs.s3a.secret.key",  "password123" ) \
    .config("spark.hadoop.fs.s3a.path.style.access", "true" ) \
    .config("spark.hadoop.fs.s3a.impl","org.apache.hadoop.fs.s3a.S3AFileSystem")
my_packages = ["org.apache.hadoop:hadoop-aws:3.3.4"]
spark = configure_spark_with_delta_pip(builder, extra_packages=my_packages).getOrCreate()

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-24cd4fbe-f44e-4325-b266-85061b1d95cf;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
downloading https://repo1.maven.org/maven2/com/amazonaws/aws-java-sdk-bundle/1.12.262/aws-java-sdk-bundle-1.12.262.jar ...
	[SUCCESSFUL ] com.amazonaws#aws-java-sdk-bundle;1.12.262!aws-java-sdk-bundle.jar (73193ms)
downloading https://repo1.maven.org/maven2/org/wildfly/openssl/wildfly-openssl/1.0.7.Final/wildfly-openssl-1.0.7.Final.jar ...
	[SUCCESSF

In [3]:
from pyspark.sql import functions as F

df = spark.range(0, 5000000) 

df = df.withColumn("category",(F.rand() * 100).cast("int")) # numeric columns

df = df.withColumn("value", F.rand() * 1000) # random distribution

In [4]:
df.rdd.getNumPartitions()

24

In [11]:
import time

start = time.time()
df.repartition(1).write \
    .format("delta") \
    .mode("overwrite") \
    .save("s3a://bronze/partition-test/repartition-1")

end = time.time()

print("Time:", end - start)

Time: 7.185253620147705


In [12]:
start = time.time()
df.repartition(5).write \
    .format("delta") \
    .mode("overwrite") \
    .save("s3a://bronze/partition-test/repartition-5")

end = time.time()

print("Time:", end - start)

Time: 3.8416006565093994


In [13]:
start = time.time()
df.repartition(20).write \
    .format("delta") \
    .mode("overwrite") \
    .save("s3a://bronze/partition-test/repartition-20")

end = time.time()

print("Time:", end - start)

Time: 4.355675220489502


In [14]:
spark.conf.get(
    "spark.sql.shuffle.partitions"
)

'200'